In [2]:
import numpy as np
import os

snapshots = [99, 50, 33, 25]

fields = [
    "SubhaloMassType",
    "SubhaloSFR",
    "SubhaloGasMetallicitySfrWeighted"
]

for snap in snapshots:

    filename = f"TNG_groupcat_data/snapshot_{snap}.npz"

    print("\n" + "=" * 60)
    print(f"SNAPSHOT {snap}")
    print("=" * 60)

    if not os.path.exists(filename):
        print("Fajl ne postoji!")
        continue

    data = np.load(filename)

    # Broj ID-jeva
    ids = data["SubhaloID"]

    print(f"\nBroj subhaloa (ID): {len(ids)}")

    # Provera svakog polja
    lengths = []

    for field in fields:

        array = data[field]

        lengths.append(len(array))

        print(f"{field}:")
        print(f"    Oblik: {array.shape}")
        print(f"    Broj redova: {len(array)}")

    # Provera jednakog broja redova
    if len(set(lengths + [len(ids)])) == 1:

        print("\nSVI NIZOVI IMAJU ISTI BROJ REDOVA.")

    else:

        print("\nGRESKA: BROJ REDOVA SE NE POKLAPA")

    # Provera ID-jeva
    expected_ids = np.arange(len(ids))

    if np.array_equal(ids, expected_ids):

        print("ID-jevi odgovaraju indeksima u katalogu.")

    else:

        print("GRESKA: ID-jevi nisu pravilno poređani")

    # Provera da li postoje duplikati
    if len(np.unique(ids)) == len(ids):

        print("Nema dupliranih ID-jeva.")

    else:

        print("GRESKA: Postoje duplirani ID-jevi.")

    # Provera nedostajućih vrednosti
    for field in fields:

        array = data[field]

        print(
            f"{field} | NaN vrednosti:",
            np.isnan(array).sum()
        )

print("\nKraj")


SNAPSHOT 99

Broj subhaloa (ID): 4371211
SubhaloMassType:
    Oblik: (4371211, 6)
    Broj redova: 4371211
SubhaloSFR:
    Oblik: (4371211,)
    Broj redova: 4371211
SubhaloGasMetallicitySfrWeighted:
    Oblik: (4371211,)
    Broj redova: 4371211

SVI NIZOVI IMAJU ISTI BROJ REDOVA.
ID-jevi odgovaraju indeksima u katalogu.
Nema dupliranih ID-jeva.
SubhaloMassType | NaN vrednosti: 0
SubhaloSFR | NaN vrednosti: 0
SubhaloGasMetallicitySfrWeighted | NaN vrednosti: 0

SNAPSHOT 50

Broj subhaloa (ID): 5127294
SubhaloMassType:
    Oblik: (5127294, 6)
    Broj redova: 5127294
SubhaloSFR:
    Oblik: (5127294,)
    Broj redova: 5127294
SubhaloGasMetallicitySfrWeighted:
    Oblik: (5127294,)
    Broj redova: 5127294

SVI NIZOVI IMAJU ISTI BROJ REDOVA.
ID-jevi odgovaraju indeksima u katalogu.
Nema dupliranih ID-jeva.
SubhaloMassType | NaN vrednosti: 0
SubhaloSFR | NaN vrednosti: 0
SubhaloGasMetallicitySfrWeighted | NaN vrednosti: 0

SNAPSHOT 33

Broj subhaloa (ID): 5621338
SubhaloMassType:
    Obl

# Filtriranje: pozitivna zvezdana masa

In [9]:
import numpy as np

snapshots = [99, 50, 33, 25]

all_data = {}

for snap in snapshots:

    filename = f"TNG_groupcat_data/snapshot_{snap}.npz"

    loaded = np.load(filename)

    all_data[snap] = {
        key: loaded[key]
        for key in loaded.files
    }

    loaded.close()

    print(f'Snapshot {snap}: učitan')

Snapshot 99: učitan
Snapshot 50: učitan
Snapshot 33: učitan
Snapshot 25: učitan


In [10]:
# Pozitivna zvezdana masa
stellar_mass = data["SubhaloMassType"][:, 4]

mask = stellar_mass > 0

In [12]:

import numpy as np

snapshots = [99, 50, 33, 25]

filtered_data = {}

for snap in snapshots:

    print("\n" + "=" * 50)
    print(f"SNAPSHOT {snap}")
    print("=" * 50)

    data = all_data[snap]

    # Ukupan broj objekata pre filtriranja
    N_before = len(data["SubhaloID"])

    # Zvezdana masa (indeks 4)
    stellar_mass = data["SubhaloMassType"][:, 4]

    # Selekcija objekata sa pozitivnom zvezdanom masom
    mask = stellar_mass > 0

    # Broj objekata nakon filtriranja
    N_after = np.sum(mask)

    # Broj izbačenih objekata
    N_removed = N_before - N_after

    # Filtriranje svih polja istom maskom
    filtered_data[snap] = {
        field: data[field][mask]
        for field in [
            "SubhaloID",
            "SubhaloMassType",
            "SubhaloSFR",
            "SubhaloGasMetallicitySfrWeighted"
        ]
    }

    print("Broj objekata pre filtriranja:", N_before)
    print("Broj objekata nakon filtriranja:", N_after)
    print("Broj izbačenih objekata:", N_removed)

    print(
        "Procenat izbačenih objekata:",
        round(100 * N_removed / N_before, 2), "%"
    )

print("\nKraj")



SNAPSHOT 99
Broj objekata pre filtriranja: 4371211
Broj objekata nakon filtriranja: 337261
Broj izbačenih objekata: 4033950
Procenat izbačenih objekata: 92.28 %

SNAPSHOT 50
Broj objekata pre filtriranja: 5127294
Broj objekata nakon filtriranja: 492655
Broj izbačenih objekata: 4634639
Procenat izbačenih objekata: 90.39 %

SNAPSHOT 33
Broj objekata pre filtriranja: 5621338
Broj objekata nakon filtriranja: 680244
Broj izbačenih objekata: 4941094
Procenat izbačenih objekata: 87.9 %

SNAPSHOT 25
Broj objekata pre filtriranja: 5828784
Broj objekata nakon filtriranja: 836537
Broj izbačenih objekata: 4992247
Procenat izbačenih objekata: 85.65 %

Kraj


# Filtriranje: logSFR>-4

In [14]:
import numpy as np

final_data = {}

for snap in snapshots:

    data = filtered_data[snap]

    # Racunanje logSFR
    logSFR = np.full(len(data["SubhaloSFR"]), -np.inf)

    positive_sfr = data["SubhaloSFR"] > 0
    logSFR[positive_sfr] = np.log10(data["SubhaloSFR"][positive_sfr])

    # Uslov: logSFR >= -4
    mask = logSFR >= -4

    # Primena maske na sva polja
    final_data[snap] = {
        key: data[key][mask]
        for key in data.keys()
    }

    N_before = len(data["SubhaloID"])
    N_after = np.sum(mask)
    N_removed = N_before - N_after

    print("=" * 50)
    print(f"SNAPSHOT {snap}")
    print("=" * 50)
    print(f"Broj objekata pre dodatnog filtriranja: {N_before}")
    print(f"Broj objekata nakon dodatnog filtriranja: {N_after}")
    print(f"Broj izbacenih objekata: {N_removed}")
    print(f"Procenat izbacenih objekata: {100*N_removed/N_before:.2f} %")

SNAPSHOT 99
Broj objekata pre dodatnog filtriranja: 337261
Broj objekata nakon dodatnog filtriranja: 71575
Broj izbacenih objekata: 265686
Procenat izbacenih objekata: 78.78 %
SNAPSHOT 50
Broj objekata pre dodatnog filtriranja: 492655
Broj objekata nakon dodatnog filtriranja: 147241
Broj izbacenih objekata: 345414
Procenat izbacenih objekata: 70.11 %
SNAPSHOT 33
Broj objekata pre dodatnog filtriranja: 680244
Broj objekata nakon dodatnog filtriranja: 274398
Broj izbacenih objekata: 405846
Procenat izbacenih objekata: 59.66 %
SNAPSHOT 25
Broj objekata pre dodatnog filtriranja: 836537
Broj objekata nakon dodatnog filtriranja: 425208
Broj izbacenih objekata: 411329
Procenat izbacenih objekata: 49.17 %
